# Variant parsing and sequence reconstruction

Use the public parsing helpers for substitutions written in DMS Parser's internal notation or as protein HGVS.

In [ ]:
import pandas as pd

from dms_parser import (
    InvalidVariantError,
    MutationApplicationError,
    UnsupportedVariantError,
    count_mutations,
    explode_mutations,
    hgvs_to_sequence,
    parse_hgvs_pro,
    parse_variant,
    parse_variant_series,
    variant_to_sequence,
)

## Internal substitution notation

`WT` has no substitutions, a single substitution looks like `M1A`, and multiple substitutions are separated by semicolons. Positions are one-based.

In [ ]:
internal_variants = pd.Series(
    ["WT", "M1A", "M1A;K2R"],
    name="variant",
)
parsed_internal = parse_variant_series(internal_variants)
parsed_internal.loc[:, ["variant", "is_wildtype", "n_mutations", "mutations"]]

In [ ]:
mutation_counts = internal_variants.map(count_mutations).rename("count")
mutation_rows = explode_mutations(parsed_internal)

mutation_counts.to_frame(), mutation_rows.loc[:, ["variant", "token", "position"]]

## Protein HGVS notation

Complete-target identity (`p.=`), equality-only brackets, single substitutions, and bracketed multi-substitutions are supported. Equality components in mixed brackets add no mutation. Protein indels and frameshifts are intentionally outside the substitutions-only contract.

In [ ]:
hgvs_variants = pd.Series(
    [
        "p.=",
        "p.[=;=]",
        "p.Met1Ala",
        "p.[=;Met1Ala]",
        "p.[Met1Ala;Lys2Arg]",
    ],
    name="hgvs_pro",
)
pd.DataFrame(
    {
        "hgvs_pro": hgvs_variants,
        "mutations": hgvs_variants.map(parse_hgvs_pro),
    }
)

In [ ]:
try:
    parse_hgvs_pro("p.Met1del")
except UnsupportedVariantError as error:
    print(error)

## Sequence reconstruction

Reconstruction validates the expected wild-type residue before applying each substitution and returns both the mutated sequence and normalized internal notation.

In [ ]:
wt_sequence = "MKTAYIAK"

reconstructed = pd.DataFrame(
    [
        ("internal", "WT", *variant_to_sequence(wt_sequence, "WT")),
        ("internal", "M1A", *variant_to_sequence(wt_sequence, "M1A")),
        ("internal", "M1A;K2R", *variant_to_sequence(wt_sequence, "M1A;K2R")),
        ("HGVS", "p.[Met1Ala;Lys2Arg]", *hgvs_to_sequence(wt_sequence, "p.[Met1Ala;Lys2Arg]")),
    ],
    columns=["notation", "input", "sequence", "normalized_variant"],
)
reconstructed

## Error handling

Catch the specific exception that corresponds to invalid syntax, unsupported variant classes, or a sequence-application failure.

In [ ]:
try:
    parse_variant("M1")
except InvalidVariantError as error:
    print(f"Invalid notation: {error}")

try:
    parse_hgvs_pro("p.Met1fs")
except UnsupportedVariantError as error:
    print(f"Unsupported variant: {error}")

try:
    variant_to_sequence(wt_sequence, "A1V")
except MutationApplicationError as error:
    print(f"Reconstruction failed: {error}")